In [0]:
use project.silver

In [0]:
CREATE TABLE   sales_delta_zorder (
  sale_id INT,
  customer_id INT,
  sale_date DATE,
  amount DOUBLE
) USING DELTA;


In [0]:
INSERT INTO sales_delta_zorder VALUES
(1, 101, '2024-01-01', 500),
(2, 102, '2024-01-02', 750),
(3, 101, '2024-01-03', 300),
(4, 103, '2024-01-04', 900),
(5, 102, '2024-01-05', 650);


num_affected_rows,num_inserted_rows
5,5


In [0]:
-- Query Before Optimization
SELECT * FROM sales_delta_zorder WHERE customer_id = 102;

sale_id,customer_id,sale_date,amount
2,102,2024-01-02,750.0
5,102,2024-01-05,650.0


In [0]:
--apply zordering
-- Databricks rewrites files.
-- similar customer_id rows stored closer.
-- Delta Lake writes statistics that help skipping data.

OPTIMIZE sales_delta_zorder
ZORDER BY (customer_id);


path,metrics
abfss://catalog@trainingbr.dfs.core.windows.net/silver/__unitystorage/schemas/f2a5427f-a7a4-4992-8af9-39d06ca9be29/tables/6074352a-b1f3-4863-a519-23339b2f688f,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 1599), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1765427410325, 1765427416508, 4, 0, null, List(0, 0), null, 4, 4, 0, 0, null)"


In [0]:
-- Query After Z-Ordering
-- Now Spark reads only a few files instead of scanning all.

SELECT * FROM sales_delta_zorder WHERE customer_id = 102;




sale_id,customer_id,sale_date,amount
2,102,2024-01-02,750.0
5,102,2024-01-05,650.0


In [0]:
%python

data = [
    (1, 101, "2024-01-01", 500),
    (2, 102, "2024-01-02", 750),
    (3, 101, "2024-01-03", 300),
    (4, 103, "2024-01-04", 900),
    (5, 102, "2024-01-05", 650),
]

cols = ["sale_id", "customer_id", "sale_date", "amount"]
df = spark.createDataFrame(data, cols)

df.write.format("delta").mode("overwrite").saveAsTable("sales_delta_pyspark")


In [0]:
%python
spark.sql("""
    OPTIMIZE sales_delta_pyspark
    ZORDER BY (customer_id)
""")


DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

Multi-Column Z-Order Example

In [0]:
OPTIMIZE events 
ZORDER BY (user_id, event_date);


In [0]:
SELECT * FROM events WHERE user_id = 10 AND event_date = '2024-01-01';
